# Анализ данных

Проверки, на которых держится решение: как устроены запросы и корпус, как строить валидацию и какие особенности данных пришлось учесть.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ноутбук запускается из папки notebooks/, данные лежат в ../data
sys.path.insert(0, str(Path.cwd().parent))
from src.data import QUERY_COLS, norm_query

pd.set_option("display.width", 200)
train = pd.read_parquet("../data/train.parquet")
bench_q = pd.read_parquet("../data/benchmark_queries.parquet")
bench_items = pd.read_parquet("../data/benchmark_items.parquet")
train["search_infm_params_text"] = train["search_infm_params_text"].fillna("")
train["text"] = norm_query(train["search_query"])
bench_q["text"] = norm_query(bench_q["search_query"])
len(train), len(bench_q), len(bench_items)

(497673, 2452, 189212)

## Запросы в train

`query_id` в train нет: строка — это пара «запрос — выбранное объявление». Запросом считаем уникальный набор признаков поиска, а все объявления, выбранные при нём, — его релевантными ответами. Полные дубли пар выкидываем.

In [2]:
print("полных дублей строк:", train.duplicated().sum())
train = train.drop_duplicates(QUERY_COLS + ["item_id"])
groups = train.groupby(QUERY_COLS).size()
print("запросов:", len(groups))
groups.clip(upper=5).value_counts(normalize=True).sort_index().round(3)

полных дублей строк: 30619


запросов: 354463


1    0.850
2    0.095
3    0.026
4    0.011
5    0.018
Name: proportion, dtype: float64

## Как устроен бенчмарк

Все тексты запросов бенчмарка уникальны, и только 37% из них встречаются в train. Если взять из train по одной случайной группе на каждый уникальный текст, доля «знакомых» текстов получается такой же. Так и строим валидацию. Запросов без фильтров в бенчмарке больше, эту долю выравниваем при сэмплировании.

In [3]:
seen = set(train["text"])
print("уникальных текстов в бенчмарке:", bench_q["text"].nunique(), "из", len(bench_q))
print("текст бенчмарка встречается в train:", round(bench_q["text"].isin(seen).mean(), 3))

g = train.groupby(QUERY_COLS + ["text"]).size().reset_index()
groups_per_text = g.groupby("text").size()
one_per_text = g.sample(frac=1, random_state=0).drop_duplicates("text")
print("одна группа на текст, у текста есть другие группы:", round((groups_per_text.reindex(one_per_text["text"]) > 1).mean(), 3))

print("без фильтров: бенчмарк", round(bench_q["search_infm_params_text"].fillna("").eq("").mean(), 3),
      "| одна группа на текст", round(one_per_text["search_infm_params_text"].eq("").mean(), 3))
print("поиск без категории (0): бенчмарк", round(bench_q["search_category"].eq(0).mean(), 3),
      "| train", round(train["search_category"].eq(0).mean(), 5))

уникальных текстов в бенчмарке: 2451 из 2452
текст бенчмарка встречается в train: 0.374


одна группа на текст, у текста есть другие группы: 0.378
без фильтров: бенчмарк 0.631 | одна группа на текст 0.474
поиск без категории (0): бенчмарк 0.091 | train 7e-05


## Корпус почти не пересекается с train

Только около 10% объявлений корпуса встречались в train. Поэтому признаков уровня конкретного объявления («сколько раз его выбирали») в решении нет: на бенчмарке они почти всегда пустые. При этом распределения подкатегорий и локаций у корпуса и у объявлений train похожи, поэтому корпус для валидации собирается из объявлений train.

In [4]:
train_items = set(train["item_id"])
print("объявлений корпуса, встречавшихся в train:", round(bench_items["item_id"].isin(train_items).mean(), 3))

def tv(a, b):
    # расстояние полной вариации: 0 — распределения совпадают, 1 — не пересекаются
    p = pd.concat([a.value_counts(normalize=True), b.value_counts(normalize=True)], axis=1).fillna(0)
    return round(0.5 * (p.iloc[:, 0] - p.iloc[:, 1]).abs().sum(), 3)

items_train = train.drop_duplicates("item_id")
print("расстояние между распределениями подкатегорий:", tv(items_train["item_microcat_id"], bench_items["item_microcat_id"]))
print("расстояние между распределениями локаций:", tv(items_train["item_location_id"], bench_items["item_location_id"]))

объявлений корпуса, встречавшихся в train: 0.096


расстояние между распределениями подкатегорий: 0.205
расстояние между распределениями локаций: 0.107


## Локации

В 82% пар объявление из той же локации, что и поиск. Но часть локаций поиска — это регионы или вся страна: они ни разу не встречаются как локация объявления, а поиск из них ведёт в города внутри. Поэтому совместимость локаций берём из статистики train, а не из равенства id.

In [5]:
print("объявление в той же локации:", round((train["search_location_id"] == train["item_location_id"]).mean(), 3))
item_locs = set(train["item_location_id"])
region_like = train.loc[~train["search_location_id"].isin(item_locs), "search_location_id"]
print("поисков из локаций, где нет объявлений:", round(len(region_like) / len(train), 3))
for loc in region_like.value_counts().index[:3]:
    top = train.loc[train["search_location_id"] == loc, "item_location_id"].value_counts(normalize=True).head(3)
    print(loc, "->", top.round(3).to_dict())

объявление в той же локации: 0.822
поисков из локаций, где нет объявлений: 0.114
107620 -> {637640: 0.611, 638790: 0.028, 637770: 0.024}
621540 -> {637640: 0.272, 653240: 0.106, 633540: 0.032}
107621 -> {653240: 0.794, 636840: 0.037, 636736: 0.018}


## Фильтры поиска

Выбранные объявления почти всегда удовлетворяют фильтрам: вид и тип услуги из фильтра есть в параметрах объявления, фильтр по рейтингу соблюдается. Из фильтров получаются сильные признаки.

In [6]:
def comply(key):
    # значение фильтра — текст после ключа до следующего слова с заглавной буквы
    rows = train[train["search_infm_params_text"].str.contains(key + " ", regex=False)]
    value = rows["search_infm_params_text"].str.extract(key + r" ([^А-ЯЁ]*[А-ЯЁ][^А-ЯЁ]*)")[0].str.strip()
    ok = [f"{key} {v}" in p for v, p in zip(value, rows["item_infm_params_text"].fillna(""))]
    return round(float(np.mean(ok)), 3), len(rows)

print("Вид услуги:", comply("Вид услуги"))
print("Тип услуги:", comply("Тип услуги"))
rating = train[train["search_infm_params_text"].str.contains("Рейтинг пользователя", regex=False)]
print("рейтинг 4+:", round((rating["item_rating"] >= 4).mean(), 3), len(rating))

Вид услуги: (0.983, 300745)


Тип услуги: (0.961, 162940)
рейтинг 4+: 0.99 838


## Особенности значений

Цены -1, 0 и 1 — заглушки вида «цена по запросу», таких почти 10%, считаем их пропуском. У части объявлений нет рейтинга и отзывов, у единиц — координат и описания. В корпусе есть около 1% объявлений не из услуг: они могут понадобиться только поиску без категории.

In [7]:
price = bench_items["item_price"].astype(float)
print("цена <= 1:", round((price <= 1).mean(), 3))
print("пропуски в корпусе:", bench_items.isna().sum()[lambda s: s > 0].to_dict())
print("объявлений не из услуг (категория != 114):", int((bench_items["item_category_id"] != 114).sum()))

цена <= 1: 0.099
пропуски в корпусе: {'item_rating_reviews_count': 11615, 'item_rating': 17631, 'item_longitude': 1, 'item_latitude': 1, 'item_description_raw': 35}
объявлений не из услуг (категория != 114): 1876
